# Platinum 4 playground — CatBoost stack + gpt-4.1 replies

Open this from the repo root or `platinum4/`. Run All to see the saved recommendation, then edit `QUESTION` and run that cell to talk to the bot. Needs `OPENAI_API_KEY` in `.env` for new replies. **2024 test is sealed.**

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

HERE = Path.cwd().resolve()
REPO = None
for p in [HERE, *HERE.parents]:
    if (p / "platinum4" / "results").exists() and (p / "src" / "gold").exists():
        REPO = p
        break
if REPO is None:
    raise FileNotFoundError("repo root not found; open from the repo or platinum4/")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.modeling.platinum4.gold_stack import (
    DEFAULT_FIXTURE,
    load_full_stack_input,
    queries_from_full_stack,
    risk_code_list,
)
from src.modeling.platinum4.pipeline import run_turn
from src.modeling.platinum4.session import empty_session, load_session, save_session

P4 = REPO / "platinum4" / "results"
ANSWERS = P4 / "answers"
SESSION_PATH = P4 / "sessions" / "playground.json"
SAVED_MD = ANSWERS / "full_stack_conversation.md"
SAVED_JSON = P4 / "full_stack_check.json"

resolved = load_full_stack_input(DEFAULT_FIXTURE)
stack = resolved["gold_stack"]
cb = stack["catboost"]
p2 = stack["platinum2"]
p3 = stack["platinum3"]
proj = stack["project"]

display(Markdown("# Platinum 4 playground — CatBoost stack + gpt-4.1 replies"))
display(Markdown(
    "We already fed **CatBoost** (not Electrum) + Platinum 2 pile + a Platinum 3 "
    f"**{p3.get('scenario')}** card into gpt-4.1. Scroll for the saved notes, then "
    "edit `QUESTION` below and re-run that cell to try your own. **2024 is sealed.**"
))

# Platinum 4 playground — CatBoost stack + gpt-4.1 replies

We already fed **CatBoost** (not Electrum) + Platinum 2 pile + a Platinum 3 **restudy** card into gpt-4.1. Scroll for the saved notes, then edit `QUESTION` below and re-run that cell to try your own. **2024 is sealed.**

## What we fed the bot

In [2]:
display(Markdown("## What we fed the bot"))
display(pd.DataFrame([
    {"slot": "project", "value": f"{proj.get('project_key')} · {proj.get('study_phase')} · {proj.get('capacity_mw')} MW · {proj.get('observation_date')}", "honest_use": "2023 val sample only"},
    {"slot": "scenario", "value": p3.get("scenario"), "honest_use": p3.get("scenario_note")},
    {"slot": "CatBoost P(quit 12m)", "value": cb.get("p_quit_12m"), "honest_use": "ranking, not a BPM trigger"},
    {"slot": "CatBoost under scenario", "value": f"{cb.get('p_quit_under_scenario')} (delta {cb.get('delta_p_quit')})", "honest_use": "sensitivity, not causal"},
    {"slot": "Platinum 2 delayed MW now", "value": p2.get("delayed_mw_now"), "honest_use": "EIA planned pile"},
    {"slot": "Platinum 2 Holt 3m / 12m", "value": f"{p2.get('delayed_mw_h3')} / {p2.get('delayed_mw_h12')}", "honest_use": "history ≤ as-of; not 2024 actuals"},
    {"slot": "GIA-matched delayed MW", "value": p2.get("gia_delayed_mw_now"), "honest_use": "thin overlay, not the full GIA list"},
    {"slot": "BPM-index risks", "value": ", ".join(risk_code_list(p3.get("risks_bpm"))) or "(none)", "honest_use": "procedure units may exist"},
    {"slot": "stub risks", "value": ", ".join(risk_code_list(p3.get("risks_stub"))) or "(none)", "honest_use": "Gaps only"},
]))
display(Markdown("Quit model is **CatBoost** (`catboost_tuned` / trial-149). Not Electrum. Not Platinum 1 delay-months."))

## What we fed the bot

,slot,value,honest_use
0,project,P::B::E291 · IA Executed · 27.0 MW · 2023-12-31,2023 val sample only
1,scenario,restudy,If one additional restudy were on the record.
2,CatBoost P(quit 12m),0.07,"ranking, not a BPM trigger"
3,CatBoost under scenario,0.055 (delta -0.0145),"sensitivity, not causal"
4,Platinum 2 delayed MW now,25299.4,EIA planned pile
5,Platinum 2 Holt 3m / 12m,27748.5 / 37044.3,history ≤ as-of; not 2024 actuals
6,GIA-matched delayed MW,0.0,"thin overlay, not the full GIA list"
7,BPM-index risks,"system_congestion, cod_already_slipped, abando...",procedure units may exist
8,stub risks,developer_serial_quit,Gaps only


Quit model is **CatBoost** (`catboost_tuned` / trial-149). Not Electrum. Not Platinum 1 delay-months.

## Saved replies

In [3]:
display(Markdown("## Saved gpt-4.1 replies (already run)"))
if not SAVED_MD.exists():
    display(Markdown(
        "_No saved thread. From the repo root run_ "
        "`python scripts/run_platinum4_full_stack.py`."
    ))
else:
    check = json.loads(SAVED_JSON.read_text(encoding="utf-8")) if SAVED_JSON.exists() else {}
    turns = (check.get("turns") or [])
    if turns:
        display(pd.DataFrame([
            {
                "turn": t.get("turn"),
                "id": t.get("id"),
                "searched": t.get("searched"),
                "shape": t.get("shape"),
                "units": ", ".join((t.get("ids") or [])[:4]),
            }
            for t in turns
        ]))
    body = SAVED_MD.read_text(encoding="utf-8")
    parts = [p.strip() for p in body.split("\n---\n") if p.strip()]
    for part in parts[1:]:
        display(Markdown(part))
    display(Markdown(f"_Source: `{SAVED_MD.relative_to(REPO)}`_"))

## Saved gpt-4.1 replies (already run)

,turn,id,searched,shape,units
0,1,t1_scoreboard,True,long,"bpm015-r33::5.4.6, bpm015-r33::6.2.11, bpm015-..."
1,2,t2_restudy_bpm,False,long,"bpm015-r33::5.4.6, bpm015-r33::4.2.4.6"
2,3,t3_both_true,False,long,"bpm015-r33::5.4.6, bpm015-r33::4.2.4.6"


## Turn 1 (t1_scoreboard)

searched=True shape=long

# Interconnection procedure note

## 1. Risk Results Track

The CatBoost model estimates a 12-month quit probability of 0.07 for this Interconnection Request (IR), based on the gold freeze (trial-149). This is a ranking score and is not a withdrawal verdict or a BPM-015 duty. Under the Platinum 3 restudy scenario, the model sensitivity shows a delta of -0.0145, meaning the quit probability would decrease if one additional restudy were on the record. This scenario delta is a sensitivity, not a causal law or a procedural trigger.

## 2. System Crowding and Delayed-MW Pile

The Platinum 2 delayed-MW pile, which measures system-wide crowding, is as follows:
- Delayed MW now: 25,299.4 MW
- Delayed MW at 3 months: 27,748.5 MW
- Delayed MW at 12 months: 37,044.3 MW

These figures represent the total EIA-reported planned MW delayed by at least 12 months versus their first-seen date, using a Holt-damped forecast. This is a system stock and does not reflect the Commercial Operation Date (COD) or delay status of this specific IR. The GIA-matched delayed-MW slice for this IR is 0.0 MW, but this overlay is thin and must not be treated as the full list of GIAs. The EIA delayed-MW pile is not a substitute for the actual GIA population.

## 3. Platinum 3 Risks: BPM-015 Index vs Stubs

The following Platinum 3 risks have a corresponding BPM-015 index:
- System congestion (BPM-015 r33 §3.1.1, §4.3)
- COD already slipped (BPM-015 r33 §5.4.6, §7.1)
- Abandonment (BPM-015 r33 §5.4.6, §6.2.11, §4.2.4.6, §5.2.3, §5.2.5, §5.3.3, §5.3.5)
- GIA execution (BPM-015 r33 §7.1)
- Restudy friction (BPM-015 r33 §5.4.6)

The following risks are stubs and do not have a BPM-015 clause in this packet:
- Developer serial quit
- FEMA hazard exposure
- IRA energy-community status

## Citations
- BPM-015 r33 §5.4.6 - Interconnection Study Restudy (pages 56-56)
- BPM-015 r33 §6.2.11 - Refunds of Definitive Planning Phase Milestones (M2, M3, M4) (pages 92-95)
- BPM-015 r33 §4.2.4.6 - Refunds of Study Deposits (pages 34-35)
- BPM-015 r33 §3.1.1 - Contour Map (pages 21-22)
- BPM-015 r33 §5.2.5 - Withdrawal from DPP Phase I (pages 50-50)
- BPM-015 r33 §4.3 - Determination of Project Linkages and Potential Grouping (pages 35-35)
- BPM-015 r33 §5.3.5 - Withdrawal from DPP Phase II (pages 52-52)
- BPM-015 r33 §5.3.3 - Interconnection Customer Decision Point II (pages 51-51)
- BPM-015 r33 §5.2.3 - Interconnection Customer Decision Point I (pages 48-48)
- BPM-015 r33 §7.1 - Suspension (pages 126-127)

## Gaps
- developer_serial_quit has no BPM-015 clause
- hazard_exposure has no BPM-015 clause
- policy_incentive has no BPM-015 clause

## Must not claim
- do not quote months of future COD slip as a model output
- do not claim a MISO Step-Up / Firm Service failure record
- do not treat EIA delayed MW as a complete GIA list (overlay is thin)
- do not treat scenario deltas as causal effects
- do not use FERC-730 as a clean delay source
- do not unseal 2024 test labels or tune on test/score

Compliance: not_determined.

## Turn 2 (t2_restudy_bpm)

searched=False shape=long

# Interconnection procedure note

## Situation
This inquiry concerns the BPM-015 r33 restudy process when MISO determines that a restudy is required because a peer interconnection request (IR) in the same Definitive Planning Phase (DPP) cycle has withdrawn before all Generator Interconnection Agreements (GIAs), Facilities Construction Agreements (FCAs), and/or Multi-Party Facilities Construction Agreements (MPFCAs) for that cycle have been executed or filed unexecuted with FERC. The CatBoost and Platinum 2/3 risk and crowding figures are context only and do not create or modify BPM-015 duties. This note restates the BPM-015 r33 restudy workflow, funding, and deadlines. It does not treat the CatBoost scenario delta as a procedural deadline and does not invent a serial-quit clause.

## Workflow

If MISO determines that a restudy of any Interconnection Study is required because an IR withdraws or is deemed withdrawn before all GIAs, FCAs, and/or MPFCAs for the DPP cycle have been executed or filed unexecuted with FERC, MISO shall provide notice of a restudy as necessary (BPM-015 r33 §5.4.6). The notice will include a preliminary analysis supporting the need for restudy, an explanation of why restudy is required, and a good faith estimate of the cost to perform the restudy.

The Interconnection Customer (IC) shall notify MISO within five (5) Business Days whether it wishes to proceed with the restudy or withdraw its IR. If the IC fails to notify MISO within this deadline, MISO will deem the IR withdrawn (BPM-015 r33 §5.4.6).

MISO will use reasonable efforts to complete the restudy within sixty (60) Calendar Days from the date of the restudy notice.

## Stakeholders

- MISO (Transmission Provider) shall issue the restudy notice, provide supporting analysis, and estimate costs.
- The Interconnection Customer (IC) shall fund the restudy from the remaining study deposit or from an additional deposit as noticed by MISO. The IC must respond within five (5) Business Days whether to proceed or withdraw.
- The Transmission Owner (TO) coordinates on affected facilities as required by the restudy.

## Citations
- BPM-015 r33 §5.4.6 - Interconnection Study Restudy (pages 56-56)

## Gaps
developer_serial_quit has no BPM-015 clause

## Must not claim
- do not quote months of future COD slip as a model output
- do not claim a MISO Step-Up / Firm Service failure record
- do not treat EIA delayed MW as a complete GIA list (overlay is thin)
- do not treat scenario deltas as causal effects
- do not use FERC-730 as a clean delay source
- do not unseal 2024 test labels or tune on test/score


Compliance: not_determined.

## Turn 3 (t3_both_true)

searched=False shape=long

# Interconnection procedure note


## 1. Platinum 2 Crowded Pile (System Crowding) Track

The Platinum 2 delayed-MW pile for this Interconnection Request (IR) is 25,299.4 MW as of the observation date, with Holt projections of 27,748.5 MW at three months and 37,044.3 MW at twelve months. This figure represents system-wide EIA planned delayed MW (delay ≥12 months versus first seen), not the commercial operation date (COD) for this IR. The GIA-matched delayed-MW slice is 0.0 MW for this IR, but this overlay is thin and must not be treated as the full list of Generator Interconnection Agreements (GIAs) in the system. The delayed-MW pile is a measure of system crowding and does not create or modify any BPM-015 duty or deadline.

## 2. Restudy Funding Clock (BPM-015 r33 §5.4.6) Track

If MISO determines that a restudy of an Interconnection Study is required because an interconnection request withdraws or is deemed withdrawn before all GIAs, Facilities Construction Agreements (FCAs), and/or Multi-Party Facilities Construction Agreements (MPFCAs) for that DPP cycle have been executed or filed unexecuted with FERC, MISO shall provide notice of a restudy as necessary. The notice will include a preliminary analysis, an explanation of why restudy is required, and a good faith estimate of the cost to perform the restudy. The Interconnection Customer (IC) shall notify MISO within five (5) Business Days whether it wishes to proceed with the restudy or withdraw the IR. Failure to notify MISO is deemed withdrawal. The IC funds the restudy from the remaining study deposit or from an additional deposit as noticed (BPM-015 r33 §5.4.6).

## 3. Can Both Be True at Once?

Yes, both tracks can be true at once without collapsing system crowding into a BPM-015 restudy trigger. The Platinum 2 delayed-MW pile and the CatBoost quit probability are risk analytics and system context. They do not create, modify, or accelerate the BPM-015 r33 §5.4.6 restudy notice, the five-Business-Day response clock, or the funding obligation. The BPM-015 restudy process is triggered only by the withdrawal or deemed withdrawal of a peer IR in the same DPP cycle, as determined by MISO, not by risk model scores or system crowding overlays.

## 4. Do CatBoost or Serial-Quit Flags Rewrite the BPM-015 Clock or Deposit?

Neither the CatBoost quit score nor the developer serial-quit flag rewrites the five-Business-Day clock or the restudy funding deposit under BPM-015 r33 §5.4.6. These analytics are for risk ranking and scenario sensitivity only. The BPM-015 process and deadlines are set by procedural events (such as peer withdrawal and MISO notice), not by risk model outputs or flags.

## Gaps
- The GIP (Attachment X) is the tariff and is not in this packet.
- developer_serial_quit has no BPM-015 clause

## Citations
- BPM-015 r33 §5.4.6 - Interconnection Study Restudy (pages 56-56)

## Must not claim
- do not quote months of future COD slip as a model output
- do not claim a MISO Step-Up / Firm Service failure record
- do not treat EIA delayed MW as a complete GIA list (overlay is thin)
- do not treat scenario deltas as causal effects
- do not use FERC-730 as a clean delay source
- do not unseal 2024 test labels or tune on test/score

Compliance: not_determined.

_Source: `platinum4\results\answers\full_stack_conversation.md`_

## Try your own question

Edit `PROJECT_KEY`, `SCENARIO`, and `QUESTION`, then run the cell. Set `RESET_SESSION = True` to start over. The next cell is a follow-up on the same thread.

In [4]:
# --- edit these, then run this cell ---
PROJECT_KEY = "P::B::E291"   # P::B::E291 | P::J2280 | P::J2460
SCENARIO = "restudy"          # baseline | restudy | enter_gia | already_past_cod | rate_shock | serial_developer | high_system_delay
QUESTION = "Does the CatBoost quit score change who funds a restudy after a peer withdrawal?"
RESET_SESSION = False        # True = start a new thread (no history)
# ---------------------------------------

from src.modeling.platinum4.errors import BotError
from src.modeling.platinum4.gold_stack import load_sample_cards, select_card, gold_stack_from_card

card = select_card(load_sample_cards(), PROJECT_KEY, SCENARIO)
stack = gold_stack_from_card(card)
query = {
    "question": QUESTION.strip(),
    "card": card,
    "gold_stack": stack,
    "requirements": {
        "audience": "analyst",
        "dialect": "auto",
        "need": ["workflow", "stakeholders", "citations", "do_not_claim"],
        "max_units": 12,
    },
    "split": "val",
}
session = empty_session() if RESET_SESSION else load_session(SESSION_PATH)
try:
    result = run_turn(query, session)
except BotError as exc:
    display(Markdown(f"**Failed** `{exc.code}`: {exc}"))
else:
    save_session(result["session"], SESSION_PATH)
    composed = result.get("composed") or {}
    md = composed.get("markdown") or ""
    ids = [u.get("unit_id") for u in (result["packet"].get("retrieved") or []) if u.get("unit_id")]
    display(Markdown(
        f"**{PROJECT_KEY}** · `{SCENARIO}` · searched=`{result.get('searched')}` · "
        f"shape=`{composed.get('shape')}` · turns in thread: "
        f"{len((result.get('session') or {}).get('turns') or [])}"
    ))
    display(Markdown(f"Units: `{', '.join(ids) or 'none'}`"))
    display(Markdown(md))
    display(Markdown(f"_Session saved to `{SESSION_PATH.relative_to(REPO)}`. Change QUESTION and re-run for a follow-up._"))

**P::B::E291** · `restudy` · searched=`False` · shape=`short` · turns in thread: 3

Units: `bpm015-r33::5.4.6, bpm015-r33::4.2.4.6`

The CatBoost quit score does not change who funds a restudy after a peer withdrawal. The Interconnection Customer (IC) is responsible for funding the Interconnection Study restudy, using any remaining study deposit or by providing an additional deposit as specified in the restudy notice from MISO (BPM-015 r33 §5.4.6). The CatBoost score is a ranking tool and does not create or modify BPM-015 duties.

## Citations
- BPM-015 r33 §5.4.6 - Interconnection Study Restudy (pages 56-56)

## Must not claim
- do not quote months of future COD slip as a model output
- do not claim a MISO Step-Up / Firm Service failure record
- do not treat EIA delayed MW as a complete GIA list (overlay is thin)


## Gaps
No additional planner gaps were recorded.

Compliance: not_determined.


_Session saved to `platinum4\results\sessions\playground.json`. Change QUESTION and re-run for a follow-up._

## Follow-up

In [5]:
# Follow-up on the same thread (uses playground.json from the cell above).
FOLLOWUP = "Who pays for that, and how many Business Days does the IC have?"

from src.modeling.platinum4.errors import BotError
from src.modeling.platinum4.gold_stack import load_sample_cards, select_card, gold_stack_from_card

card = select_card(load_sample_cards(), PROJECT_KEY, SCENARIO)
query = {
    "question": FOLLOWUP.strip(),
    "card": card,
    "gold_stack": gold_stack_from_card(card),
    "requirements": {
        "audience": "analyst",
        "dialect": "auto",
        "need": ["workflow", "citations", "do_not_claim"],
        "max_units": 12,
    },
    "split": "val",
}
session = load_session(SESSION_PATH)
try:
    result = run_turn(query, session)
except BotError as exc:
    display(Markdown(f"**Failed** `{exc.code}`: {exc}"))
except NameError:
    display(Markdown("Run the cell above first so `PROJECT_KEY` / `SCENARIO` exist."))
else:
    save_session(result["session"], SESSION_PATH)
    composed = result.get("composed") or {}
    display(Markdown(
        f"Follow-up searched=`{result.get('searched')}` · shape=`{composed.get('shape')}`"
    ))
    display(Markdown(composed.get("markdown") or "_empty_"))

Follow-up searched=`False` · shape=`short`

The Interconnection Customer (IC) is responsible for funding the Interconnection Study restudy, using any remaining study deposit or by providing an additional deposit as specified in the restudy notice from MISO (BPM-015 r33 §5.4.6). The IC must notify MISO within five (5) Business Days whether it wishes to proceed with the restudy or withdraw its interconnection request (IR) (BPM-015 r33 §5.4.6).

## Citations
- BPM-015 r33 §5.4.6 - Interconnection Study Restudy (pages 56-56)

## Must not claim
- do not quote months of future COD slip as a model output
- do not claim a MISO Step-Up / Firm Service failure record
- do not treat EIA delayed MW as a complete GIA list (overlay is thin)


## Gaps
No additional planner gaps were recorded.

Compliance: not_determined.


In [6]:
display(Markdown("## What this is not"))
display(Markdown(
    "- Not Electrum. Quit scores are **CatBoost**.\n"
    "- Not Platinum 1 months of COD slip.\n"
    "- Not a FERC compliance verdict (`compliance` is `not_determined`).\n"
    "- `developer_serial_quit`, `hazard_exposure`, `policy_incentive` have no BPM clause.\n"
    "- Do not pass `split=test`. 2024 stays sealed."
))
display(Markdown(
    "Replay the canned 3-turn thread: `python scripts/run_platinum4_full_stack.py`  \n"
    "Rebuild this notebook: `python scripts/build_platinum4_playground.py`"
))

## What this is not

- Not Electrum. Quit scores are **CatBoost**.
- Not Platinum 1 months of COD slip.
- Not a FERC compliance verdict (`compliance` is `not_determined`).
- `developer_serial_quit`, `hazard_exposure`, `policy_incentive` have no BPM clause.
- Do not pass `split=test`. 2024 stays sealed.

Replay the canned 3-turn thread: `python scripts/run_platinum4_full_stack.py`  
Rebuild this notebook: `python scripts/build_platinum4_playground.py`